# Stokes theorem on a spherical surface

This tutorial demonstrates line and surface integration with LowLevelFEM by verifying Stokes' theorem

$$\oint_{\partial S} \mathbf{v}\cdot\mathbf{t}\,\mathrm{d}s = \int_S (\nabla\times\mathbf{v})\cdot\mathbf{n}\,\mathrm{d}A.$$

The surface $S$ is one octant of a sphere. Its boundary $\partial S$ consists of three quarter-circle arcs. The orientation of the boundary tangent $\mathbf{t}$ is consistent with the outward surface normal $\mathbf{n}$.

## Load the packages and initialize Gmsh

`LinearAlgebra` supplies the familiar vector-product notation used by the LowLevelFEM field operations.

In [1]:
using LowLevelFEM, LinearAlgebra

gmsh.initialize()

## Create the spherical-octant mesh

The radius is passed to `sphere-part.geo`. The geometry file creates a second-order tetrahedral mesh and defines `"volu"` for the spherical octant, `"surf"` for its curved surface, and `"peri"` for the three boundary arcs.

In [2]:
R = 10.0

setParameter("R", R)
gmsh.merge("sphere-part.geo")

Info    : Reading 'sphere-part.geo'...
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Circle)
Info    : [ 20%] Meshing curve 2 (Circle)
Info    : [ 40%] Meshing curve 3 (Circle)
Info    : [ 60%] Meshing curve 4 (Line)
Info    : [ 70%] Meshing curve 5 (Line)
Info    : [ 90%] Meshing curve 6 (Line)
Info    : Done meshing 1D (Wall 0.000952591s, CPU 0.000893s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Surface, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 2 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 3 (Plane, Frontal-Delaunay)
Info    : [ 80%] Meshing surface 4 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0313366s, CPU 0.030993s)
Info    : Meshing 3D...
Info    : 3D Meshing 1 volume with 1 connected component
Info    : Tetrahedrizing 498 nodes...
Info    : Done tetrahedrizing 506 nodes (Wall 0.00823646s, CPU 0.008159s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges


## Construct a LowLevelFEM problem

The example does not solve a finite element equation. The `Problem` object gives the field and integration routines access to the mesh and its physical groups.

In [3]:
mat = material("volu")
prob = Problem([mat]);

## Define the vector field

A smooth three-dimensional vector field is specified component by component with ordinary Julia functions.

In [4]:
vx(x, y, z) = x * y * z / 100
vy(x, y, z) = (x^2 + y^2 + z^2) / 10
vz(x, y, z) = (2x * y + 2y * z + 2z) / 10;

The oriented tangent and normal fields are generated from the boundary curves and the spherical surface. The analytical vector field is sampled on all three relevant physical groups.

In [5]:
t = tangentVector(prob, "peri")
n = normalVector(prob, "surf")
v_peri = VectorField(prob, "peri", [vx, vy, vz])
v_surf = VectorField(prob, "surf", [vx, vy, vz])
v_volu = VectorField(prob, "volu", [vx, vy, vz]);

## Evaluate the circulation

The line integral sums the tangential component of $\mathbf{v}$ along the three oriented boundary arcs.

In [6]:
circulation = integrate(prob, "peri", v_peri ⋅ t)

66.66670977826737

## Evaluate the curl flux

The full three-dimensional curl requires derivatives in all three spatial directions. It is therefore computed from the field defined in the volume. The nodal values are then transferred to the elements of the spherical surface before the flux is integrated. Computing the curl directly from a field defined only on the two-dimensional surface would not provide the missing normal derivative.

In [7]:
curl_v = ∇ × v_volu
curl_on_surf = nodesToElements(elementsToNodes(curl_v), onPhysicalGroup="surf")
curl_flux = integrate(prob, "surf", curl_on_surf ⋅ n)
relative_difference = abs(circulation - curl_flux) / abs(circulation)

(; circulation, curl_flux, relative_difference)

(circulation = 66.66670977826737, curl_flux = 66.65207550398274, relative_difference = 0.00021951397231549145)

The circulation and curl flux should agree up to the discretization, interpolation, and quadrature errors.

## Integrate constants: boundary length and surface area

Integrating the constant function $1$ measures the selected curve length or surface area. The boundary contains three quarter-circles, while the curved surface is one eighth of a sphere:

$$L = 3\frac{\pi R}{2}, \qquad A = \frac{4\pi R^2}{8} = \frac{\pi R^2}{2}.$$

In [8]:
boundary_length = integrate(prob, "peri", (x, y, z) -> 1.0)
exact_boundary_length = 3π * R / 2

surface_area = integrate(prob, "surf", (x, y, z) -> 1.0)
exact_surface_area = π * R^2 / 2

(; boundary_length, exact_boundary_length, surface_area, exact_surface_area)

(boundary_length = 47.12388828415763, exact_boundary_length = 47.12388980384689, surface_area = 157.07950054002134, exact_surface_area = 157.07963267948966)

## Inspect the fields in the Gmsh post-processor

These views make the orientation and the integrands visible on the corresponding curve, surface, and volume elements.

In [9]:
showElementResults(t, name="boundary tangent")
showElementResults(n, name="surface normal")
showElementResults(v_peri, name="v on boundary")
showElementResults(v_surf, name="v on surface")
showElementResults(v_peri ⋅ t, name="tangential component")
showElementResults(curl_on_surf, name="curl on surface")

6

## Additional check: curl of a rigid-body rotation field

For a constant angular velocity $\boldsymbol{\omega}$ and position vector $\mathbf{r}$, the rigid-body velocity is $\mathbf{v}=\boldsymbol{\omega}\times\mathbf{r}$. Its curl satisfies

$$\frac{1}{2}\nabla\times\mathbf{v}=\boldsymbol{\omega}.$$

This provides a compact independent check of the vector-field cross product and curl operator.

In [10]:
position = VectorField(prob, "volu", [(x, y, z) -> x, (x, y, z) -> y, (x, y, z) -> z])
omega = VectorField(prob, "volu", [1.0, 2.0, 3.0])
rigid_velocity = omega × position
recovered_omega = (∇ × rigid_velocity) / 2

elementwise VectorField
[[1.0000000000000018; 2.0; … ; 2.0; 3.0;;], [0.9999999999999996; 2.000000000000001; … ; 2.0000000000000004; 3.0000000000000004;;], [1.0000000000000036; 1.9999999999999982; … ; 1.999999999999999; 3.0000000000000018;;], [1.0000000000000004; 2.0000000000000018; … ; 1.9999999999999978; 2.9999999999999996;;], [0.999999999999999; 1.999999999999997; … ; 1.9999999999999996; 2.9999999999999982;;], [1.0; 1.9999999999999991; … ; 2.0000000000000018; 3.0000000000000036;;], [1.0; 2.0000000000000013; … ; 2.0000000000000027; 2.9999999999999996;;], [1.0000000000000036; 1.9999999999999938; … ; 2.0; 3.0;;], [1.0000000000000036; 2.0000000000000027; … ; 2.000000000000001; 3.0;;], [1.0000000000000018; 1.9999999999999973; … ; 2.0000000000000027; 3.0000000000000018;;]  …  [1.0000000000000018; 1.9999999999999964; … ; 1.9999999999999987; 3.0000000000000018;;], [1.0000000000000018; 2.000000000000007; … ; 2.0000000000000036; 3.000000000000007;;], [0.9999999999999964; 1.999999999999993; … ;

In [11]:
showDoFResults(omega, name="prescribed omega")
showDoFResults(rigid_velocity, name="rigid-body velocity")
showDoFResults(recovered_omega, name="recovered omega")

openPostProcessor()

-------------------------------------------------------
Version       : 4.15.2-git
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.8
OCC version   : 7.9.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


XOpenIM() failed
Fontconfig warning: using without calling FcInit()


## Finalize Gmsh

Close the Gmsh API after finishing the calculations and visualization.

In [12]:
gmsh.finalize()